
# Step 4C — External Replication on a Second Diffusion Model / Dataset

**Goal:** test whether the residual-complexity phenomenon survives outside CIFAR-10.

Second model:
- `anton-l/ddpm-ema-flowers-64`
- unconditional DDPM
- Flowers-102 domain
- 64×64 RGB images

We intentionally keep the replication simple:
- fixed deterministic reference: **DDIM-500**
- baselines: **DDIM-{20, 50, 100, 200}**
- same input noise for all maps
- same correction architecture family as the CIFAR experiment
- population proxy: held-out normalized MSE (NMSE)

Important: DDIM-500 is a **chosen deterministic high-budget reference map**, not claimed to be an exact continuous-time solution.


In [1]:

from pathlib import Path
import copy
import math
import gc
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from diffusers import DDPMPipeline, DDIMScheduler

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 20260925

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


Device: cuda
GPU: NVIDIA GeForce RTX 2060
CUDA: 12.1


## 1. Load the second pretrained diffusion model

In [2]:

MODEL_ID = "anton-l/ddpm-ema-flowers-64"

pipe = DDPMPipeline.from_pretrained(MODEL_ID)

unet = pipe.unet.to(device)
unet.eval()

base_scheduler_config = pipe.scheduler.config

sample_size = unet.config.sample_size
if isinstance(sample_size, (tuple, list)):
    H, W = int(sample_size[0]), int(sample_size[1])
else:
    H = W = int(sample_size)

IN_CHANNELS = int(unet.config.in_channels)

print("Model:", MODEL_ID)
print("Resolution:", H, "x", W)
print("Channels:", IN_CHANNELS)


model_index.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

C:\Users\User\anaconda3\envs\tabpfn_env\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--anton-l--ddpm-ema-flowers-64. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch C:\Users\User\.cache\huggingface\hub\models--anton-l--ddpm-ema-flowers-64\snapshots\dc3af58142513ad2e41e94b57a35ec3a04286a94\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\User\.cache\huggingface\hub\models--anton-l--ddpm-ema-flowers-64\snapshots\dc3af58142513ad2e41e94b57a35ec3a04286a94\unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Model: anton-l/ddpm-ema-flowers-64
Resolution: 64 x 64
Channels: 3



## 2. Experimental configuration

We use **N = 256** independent noise inputs:
- 192 train
- 32 validation
- 32 test

If GPU memory is tight, lower `GEN_BATCH` from 4 to 2 or 1.  
Do **not** lower `N_TOTAL` unless necessary, because the purpose of 4C is a meaningful external replication.


In [3]:

OUT_DIR = Path("flowers64_step4c")
OUT_DIR.mkdir(exist_ok=True)

N_TOTAL = 256
N_TRAIN = 192
N_VAL = 32
N_TEST = 32

assert N_TRAIN + N_VAL + N_TEST == N_TOTAL

REFERENCE_STEPS = 500
C_LIST = [20, 50, 100, 200]

# RTX 2050-safe default. Reduce to 2 or 1 if needed.
GEN_BATCH = 4

# Correction family
WIDTHS = [1, 2, 4, 8, 16]

print("Reference:", f"DDIM-{REFERENCE_STEPS}")
print("Baselines:", C_LIST)
print("N:", N_TOTAL)
print("Split:", N_TRAIN, N_VAL, N_TEST)


Reference: DDIM-500
Baselines: [20, 50, 100, 200]
N: 256
Split: 192 32 32


## 3. Create / restore the fixed noise dataset

In [4]:

Z_PATH = OUT_DIR / "z.pt"

if Z_PATH.exists():
    z_all = torch.load(Z_PATH, map_location="cpu")
    print("Loaded existing z:", tuple(z_all.shape))
else:
    g = torch.Generator(device="cpu")
    g.manual_seed(SEED)

    z_all = torch.randn(
        N_TOTAL,
        IN_CHANNELS,
        H,
        W,
        generator=g,
        dtype=torch.float32
    )

    torch.save(z_all, Z_PATH)
    print("Generated and saved z:", tuple(z_all.shape))

assert z_all.shape == (N_TOTAL, IN_CHANNELS, H, W)
assert torch.isfinite(z_all).all()


Generated and saved z: (256, 3, 64, 64)


## 4. Deterministic DDIM map

In [5]:

@torch.no_grad()
def ddim_map_batch(z, num_steps):
    scheduler = DDIMScheduler.from_config(base_scheduler_config)
    scheduler.set_timesteps(
        num_inference_steps=num_steps,
        device=device
    )

    x = z.to(device, dtype=torch.float32)

    for t in scheduler.timesteps:
        model_input = scheduler.scale_model_input(x, t)

        noise_pred = unet(
            model_input,
            t
        ).sample

        x = scheduler.step(
            model_output=noise_pred,
            timestep=t,
            sample=x,
            eta=0.0
        ).prev_sample

        if not torch.isfinite(x).all():
            raise FloatingPointError(
                f"NaN/Inf detected during DDIM-{num_steps} at timestep {int(t)}."
            )

    return x.detach().cpu()


@torch.no_grad()
def ddim_map_dataset(z, num_steps, batch_size=4):
    outputs = []

    for start in range(0, len(z), batch_size):
        end = min(start + batch_size, len(z))

        print(
            f"DDIM-{num_steps}: "
            f"{start:03d}:{end:03d} / {len(z)}"
        )

        out = ddim_map_batch(
            z[start:end],
            num_steps=num_steps
        )

        outputs.append(out)

        if device == "cuda":
            torch.cuda.empty_cache()

    result = torch.cat(outputs, dim=0)

    assert result.shape == z.shape
    assert torch.isfinite(result).all()

    return result


def load_or_generate_map(z, num_steps, label):
    path = OUT_DIR / f"{label}.pt"

    if path.exists():
        x = torch.load(path, map_location="cpu")

        valid = (
            tuple(x.shape) == tuple(z.shape)
            and torch.isfinite(x).all().item()
        )

        if valid:
            print(f"Loaded {path}")
            return x

        print(f"Existing {path} is invalid; regenerating.")

    x = ddim_map_dataset(
        z,
        num_steps=num_steps,
        batch_size=GEN_BATCH
    )

    torch.save(x, path)
    print(f"Saved {path}")

    return x


## 5. Generate / restore the reference and baselines

In [6]:

teacher = load_or_generate_map(
    z_all,
    num_steps=REFERENCE_STEPS,
    label=f"teacher_ddim_{REFERENCE_STEPS}"
)

baselines = {}

for C in C_LIST:
    baselines[C] = load_or_generate_map(
        z_all,
        num_steps=C,
        label=f"baseline_ddim_{C}"
    )

print("\nAll maps ready.")

for C in C_LIST:
    print(
        C,
        tuple(baselines[C].shape),
        "finite =",
        torch.isfinite(baselines[C]).all().item()
    )


DDIM-500: 000:004 / 256
DDIM-500: 004:008 / 256
DDIM-500: 008:012 / 256
DDIM-500: 012:016 / 256
DDIM-500: 016:020 / 256
DDIM-500: 020:024 / 256
DDIM-500: 024:028 / 256
DDIM-500: 028:032 / 256
DDIM-500: 032:036 / 256
DDIM-500: 036:040 / 256
DDIM-500: 040:044 / 256
DDIM-500: 044:048 / 256
DDIM-500: 048:052 / 256
DDIM-500: 052:056 / 256
DDIM-500: 056:060 / 256
DDIM-500: 060:064 / 256
DDIM-500: 064:068 / 256
DDIM-500: 068:072 / 256
DDIM-500: 072:076 / 256
DDIM-500: 076:080 / 256
DDIM-500: 080:084 / 256
DDIM-500: 084:088 / 256
DDIM-500: 088:092 / 256
DDIM-500: 092:096 / 256
DDIM-500: 096:100 / 256
DDIM-500: 100:104 / 256
DDIM-500: 104:108 / 256
DDIM-500: 108:112 / 256
DDIM-500: 112:116 / 256
DDIM-500: 116:120 / 256
DDIM-500: 120:124 / 256
DDIM-500: 124:128 / 256
DDIM-500: 128:132 / 256
DDIM-500: 132:136 / 256
DDIM-500: 136:140 / 256
DDIM-500: 140:144 / 256
DDIM-500: 144:148 / 256
DDIM-500: 148:152 / 256
DDIM-500: 152:156 / 256
DDIM-500: 156:160 / 256
DDIM-500: 160:164 / 256
DDIM-500: 164:16

## 6. Residual magnitude

In [7]:

residuals = {}
mag_rows = []

teacher_energy = (teacher ** 2).mean().item()

for C in C_LIST:
    R = teacher - baselines[C]
    residuals[C] = R

    mse = (R ** 2).mean().item()
    rms = mse ** 0.5

    mag_rows.append({
        "C": C,
        "residual_mse": mse,
        "residual_rms": rms,
        "relative_to_teacher_energy": mse / teacher_energy
    })

mag_df = pd.DataFrame(mag_rows).sort_values("C").reset_index(drop=True)

display(mag_df)

mag_df.to_csv(
    OUT_DIR / "residual_magnitude.csv",
    index=False
)


,C,residual_mse,residual_rms,relative_to_teacher_energy
0,20,0.206994,0.454966,1.101546
1,50,0.161740,0.402169,0.860717
2,100,0.116402,0.341178,0.619448
3,200,0.073846,0.271746,0.392981



## 7. Fixed train / validation / test split

The split is shared across every baseline so that comparisons are paired.


In [8]:

train_end = N_TRAIN
val_end = N_TRAIN + N_VAL

z_train = z_all[:train_end]
z_val = z_all[train_end:val_end]
z_test = z_all[val_end:]

print("z_train:", tuple(z_train.shape))
print("z_val:  ", tuple(z_val.shape))
print("z_test: ", tuple(z_test.shape))


z_train: (192, 3, 64, 64)
z_val:   (32, 3, 64, 64)
z_test:  (32, 3, 64, 64)


## 8. Correction model

In [9]:

class CorrectionCNN(nn.Module):
    def __init__(self, width):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1),
            nn.GELU(),

            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),

            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),

            nn.Conv2d(width, 3, 3, padding=1)
        )

    def forward(self, x):
        return self.net(x)


def count_params(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


for w in WIDTHS:
    model = CorrectionCNN(w)
    print(f"width={w:2d} params={count_params(model):6d}")


width= 1 params=    78
width= 2 params=   189
width= 4 params=   519
width= 8 params=  1611
width=16 params=  5523


## 9. Robust trainer with train-only normalization and early stopping

In [10]:

def train_corrector_v2(
    z_train,
    r_train,
    z_val,
    r_val,
    z_test,
    r_test,
    width,
    max_epochs=800,
    batch_size=16,
    lr=3e-4,
    patience=60,
    seed=0
):
    torch.manual_seed(seed)

    model = CorrectionCNN(width).to(device)

    # -------------------------
    # Finite checks
    # -------------------------
    tensors = {
        "z_train": z_train,
        "r_train": r_train,
        "z_val": z_val,
        "r_val": r_val,
        "z_test": z_test,
        "r_test": r_test,
    }

    for name, tensor in tensors.items():
        if not torch.isfinite(tensor).all():
            raise ValueError(f"{name} contains NaN/Inf.")

    train_energy = (r_train ** 2).mean().item()

    if train_energy <= 1e-14:
        raise ValueError("Train residual energy is effectively zero.")

    train_rms = train_energy ** 0.5
    y_train = r_train / train_rms

    dataset = TensorDataset(z_train, y_train)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    loss_fn = nn.MSELoss()

    # -------------------------
    # Initial validation state
    # -------------------------
    model.eval()

    val_energy = (r_val ** 2).mean().item()

    if val_energy <= 1e-14:
        raise ValueError("Validation residual energy is effectively zero.")

    with torch.no_grad():
        pred_val = (
            model(z_val.to(device)).cpu()
            * train_rms
        )

    initial_val_nmse = (
        ((pred_val - r_val) ** 2).mean().item()
        / val_energy
    )

    if not math.isfinite(initial_val_nmse):
        raise ValueError(
            f"Initial validation NMSE is non-finite: {initial_val_nmse}"
        )

    best_val_nmse = initial_val_nmse
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    bad_epochs = 0

    # -------------------------
    # Training
    # -------------------------
    for epoch in range(max_epochs):
        model.train()

        epoch_ok = True

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            pred = model(xb)
            loss = loss_fn(pred, yb)

            if not torch.isfinite(loss):
                epoch_ok = False
                break

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

        if not epoch_ok:
            print(
                f"Non-finite training loss at epoch {epoch+1}; stopping."
            )
            break

        model.eval()

        with torch.no_grad():
            pred_val = (
                model(z_val.to(device)).cpu()
                * train_rms
            )

        val_nmse = (
            ((pred_val - r_val) ** 2).mean().item()
            / val_energy
        )

        if not math.isfinite(val_nmse):
            print(
                f"Non-finite validation NMSE at epoch {epoch+1}; stopping."
            )
            break

        if val_nmse < best_val_nmse - 1e-5:
            best_val_nmse = val_nmse
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_train = (
            model(z_train.to(device)).cpu()
            * train_rms
        )

        pred_test = (
            model(z_test.to(device)).cpu()
            * train_rms
        )

    train_nmse = (
        ((pred_train - r_train) ** 2).mean().item()
        / (r_train ** 2).mean().item()
    )

    test_nmse = (
        ((pred_test - r_test) ** 2).mean().item()
        / (r_test ** 2).mean().item()
    )

    result = {
        "train_nmse": train_nmse,
        "val_nmse": best_val_nmse,
        "test_nmse": test_nmse,
        "best_epoch": best_epoch,
        "params": count_params(model),
        "train_rms": train_rms,
    }

    del model
    gc.collect()

    if device == "cuda":
        torch.cuda.empty_cache()

    return result


print("Trainer ready.")


Trainer ready.


## 10. One diagnostic run before the full grid

In [11]:

C_DIAG = 50

R = residuals[C_DIAG]

diag = train_corrector_v2(
    z_train,
    R[:train_end],
    z_val,
    R[train_end:val_end],
    z_test,
    R[val_end:],
    width=4,
    max_epochs=300,
    patience=40,
    seed=0
)

print(diag)


{'train_nmse': 0.8051731969312556, 'val_nmse': 0.8179637190903759, 'test_nmse': 0.8045452194040493, 'best_epoch': 300, 'params': 519, 'train_rms': 0.39715414367880836}



## 11. Full residual learnability grid

This is the main 4C experiment.

The CSV is updated after each run, so if the notebook is interrupted you still keep completed results.


In [12]:

RESULTS_PATH = OUT_DIR / "learnability_results.csv"

if RESULTS_PATH.exists():
    results_df_existing = pd.read_csv(RESULTS_PATH)
    learning_results = results_df_existing.to_dict("records")
    print(f"Loaded {len(learning_results)} existing rows.")
else:
    learning_results = []


def already_done(C, width, seed=0):
    for row in learning_results:
        if (
            int(row["C"]) == int(C)
            and int(row["width"]) == int(width)
            and int(row["seed"]) == int(seed)
        ):
            return True
    return False


for C in C_LIST:
    R = residuals[C]

    r_train = R[:train_end]
    r_val = R[train_end:val_end]
    r_test = R[val_end:]

    for width in WIDTHS:
        if already_done(C, width, seed=0):
            print(f"SKIP existing: C={C}, width={width}")
            continue

        print("\n" + "=" * 60)
        print(f"Training C={C}, width={width}")

        out = train_corrector_v2(
            z_train,
            r_train,
            z_val,
            r_val,
            z_test,
            r_test,
            width=width,
            max_epochs=800,
            patience=60,
            seed=0
        )

        row = {
            "C": C,
            "width": width,
            "params": out["params"],
            "seed": 0,
            "train_nmse": out["train_nmse"],
            "val_nmse": out["val_nmse"],
            "test_nmse": out["test_nmse"],
            "best_epoch": out["best_epoch"],
            "train_rms": out["train_rms"],
        }

        learning_results.append(row)

        pd.DataFrame(learning_results).to_csv(
            RESULTS_PATH,
            index=False
        )

        print(row)

df_learning = (
    pd.DataFrame(learning_results)
    .sort_values(["C", "params", "seed"])
    .reset_index(drop=True)
)

display(df_learning)



Training C=20, width=1
{'C': 20, 'width': 1, 'params': 78, 'seed': 0, 'train_nmse': 0.88566164289091, 'val_nmse': 0.8911999352719135, 'test_nmse': 0.8835523988928663, 'best_epoch': 789, 'train_rms': 0.4460492648042201}

Training C=20, width=2
{'C': 20, 'width': 2, 'params': 189, 'seed': 0, 'train_nmse': 0.8557990967029899, 'val_nmse': 0.8485961860644388, 'test_nmse': 0.8695393559408199, 'best_epoch': 471, 'train_rms': 0.4460492648042201}

Training C=20, width=4
{'C': 20, 'width': 4, 'params': 519, 'seed': 0, 'train_nmse': 0.8233922829100352, 'val_nmse': 0.8132600089404997, 'test_nmse': 0.8486454615363102, 'best_epoch': 422, 'train_rms': 0.4460492648042201}

Training C=20, width=8
{'C': 20, 'width': 8, 'params': 1611, 'seed': 0, 'train_nmse': 0.8250947275519369, 'val_nmse': 0.8014278816636491, 'test_nmse': 0.8570409121205799, 'best_epoch': 260, 'train_rms': 0.4460492648042201}

Training C=20, width=16
{'C': 20, 'width': 16, 'params': 5523, 'seed': 0, 'train_nmse': 0.81798589212563, 'va

,C,width,params,seed,train_nmse,val_nmse,test_nmse,best_epoch,train_rms
0,20,1,78,0,0.885662,0.891200,0.883552,789,0.446049
1,20,2,189,0,0.855799,0.848596,0.869539,471,0.446049
2,20,4,519,0,0.823392,0.813260,0.848645,422,0.446049
3,20,8,1611,0,0.825095,0.801428,0.857041,260,0.446049
4,20,16,5523,0,0.817986,0.798349,0.850891,165,0.446049
5,50,1,78,0,0.877088,0.890267,0.863957,789,0.397154
6,50,2,189,0,0.834344,0.836942,0.836532,532,0.397154
7,50,4,519,0,0.797620,0.805630,0.802100,539,0.397154
8,50,8,1611,0,0.807698,0.795985,0.823303,266,0.397154
9,50,16,5523,0,0.803274,0.787961,0.822729,165,0.397154


## 12. Main NMSE table

In [13]:

nmse_table = df_learning.pivot_table(
    index="width",
    columns="C",
    values="test_nmse",
    aggfunc="mean"
)

display(nmse_table)


C,20,50,100,200
width,,,,
1,0.883552,0.863957,0.859624,0.888756
2,0.869539,0.836532,0.809430,0.821853
4,0.848645,0.802100,0.778089,0.794538
8,0.857041,0.823303,0.783139,0.784017
16,0.850891,0.822729,0.771138,0.776834


## 13. Empirical RAC table

In [14]:

RHO_LIST = [0.20, 0.30, 0.35, 0.40, 0.50]

rac_rows = []

for rho in RHO_LIST:
    for C in C_LIST:
        sub = (
            df_learning[df_learning["C"] == C]
            .sort_values("params")
        )

        feasible = sub[sub["test_nmse"] <= rho]

        if len(feasible) == 0:
            min_params = np.nan
            achieved_nmse = np.nan
            width = np.nan
        else:
            best = feasible.iloc[0]
            min_params = int(best["params"])
            achieved_nmse = float(best["test_nmse"])
            width = int(best["width"])

        rac_rows.append({
            "rho": rho,
            "C": C,
            "min_params": min_params,
            "width": width,
            "achieved_test_nmse": achieved_nmse
        })

rac_df = pd.DataFrame(rac_rows)

rac_pivot = rac_df.pivot(
    index="rho",
    columns="C",
    values="min_params"
)

display(rac_pivot)

rac_df.to_csv(
    OUT_DIR / "empirical_rac.csv",
    index=False
)


C,20,50,100,200
rho,,,,
0.20,NaN,NaN,NaN,NaN
0.30,NaN,NaN,NaN,NaN
0.35,NaN,NaN,NaN,NaN
0.40,NaN,NaN,NaN,NaN
0.50,NaN,NaN,NaN,NaN



## 14. Automatic search for magnitude–difficulty inversions

At a fixed corrector width, an inversion is a pair \(C_a,C_b\) such that:

\[
D(B_{C_b}) < D(B_{C_a})
\]

but

\[
\widehat E_{B_{C_b}}(K) > \widehat E_{B_{C_a}}(K).
\]

That is the empirical pattern predicted by the paper's separation story.


In [15]:

mag_map = dict(
    zip(
        mag_df["C"],
        mag_df["residual_mse"]
    )
)

inversions = []

for width in WIDTHS:
    sub = (
        df_learning[df_learning["width"] == width]
        .set_index("C")
    )

    for Ca in C_LIST:
        for Cb in C_LIST:
            if Ca >= Cb:
                continue

            mse_a = mag_map[Ca]
            mse_b = mag_map[Cb]

            nmse_a = float(sub.loc[Ca, "test_nmse"])
            nmse_b = float(sub.loc[Cb, "test_nmse"])

            # Cb has smaller absolute residual but higher normalized difficulty
            if (mse_b < mse_a) and (nmse_b > nmse_a):
                inversions.append({
                    "width": width,
                    "params": int(sub.loc[Ca, "params"]),
                    "C_easy_abs": Cb,
                    "C_large_abs": Ca,
                    "mse_smaller": mse_b,
                    "mse_larger": mse_a,
                    "nmse_harder": nmse_b,
                    "nmse_easier": nmse_a,
                    "nmse_gap": nmse_b - nmse_a
                })

inv_df = pd.DataFrame(inversions)

if len(inv_df) == 0:
    print("No magnitude–difficulty inversion found in the tested grid.")
else:
    inv_df = inv_df.sort_values(
        "nmse_gap",
        ascending=False
    ).reset_index(drop=True)

    display(inv_df.head(20))

    inv_df.to_csv(
        OUT_DIR / "magnitude_difficulty_inversions.csv",
        index=False
    )


,width,params,C_easy_abs,C_large_abs,mse_smaller,mse_larger,nmse_harder,nmse_easier,nmse_gap
0,1,78,200,100,0.073846,0.116402,0.888756,0.859624,0.029132
1,1,78,200,50,0.073846,0.161740,0.888756,0.863957,0.024799
2,4,519,200,100,0.073846,0.116402,0.794538,0.778089,0.016449
3,2,189,200,100,0.073846,0.116402,0.821853,0.809430,0.012423
4,16,5523,200,100,0.073846,0.116402,0.776834,0.771138,0.005696
5,1,78,200,20,0.073846,0.206994,0.888756,0.883552,0.005203
6,8,1611,200,100,0.073846,0.116402,0.784017,0.783139,0.000878


## 15. Compact summary to send back

In [16]:

best_by_C = (
    df_learning
    .groupby("C", as_index=False)
    .agg(
        best_test_nmse=("test_nmse", "min"),
        mean_test_nmse=("test_nmse", "mean"),
        min_train_nmse=("train_nmse", "min"),
        max_best_epoch=("best_epoch", "max")
    )
)

summary = (
    mag_df
    .merge(best_by_C, on="C", how="left")
    .sort_values("C")
)

display(summary)

print("\n=== NMSE TABLE ===")
display(nmse_table)

print("\n=== EMPIRICAL RAC ===")
display(rac_pivot)

print("\n=== TOP INVERSIONS ===")
if len(inv_df) > 0:
    display(inv_df.head(10))
else:
    print("None found.")


,C,residual_mse,residual_rms,relative_to_teacher_energy,best_test_nmse,mean_test_nmse,min_train_nmse,max_best_epoch
0,20,0.206994,0.454966,1.101546,0.848645,0.861934,0.817986,789
1,50,0.161740,0.402169,0.860717,0.802100,0.829724,0.797620,789
2,100,0.116402,0.341178,0.619448,0.771138,0.800284,0.764606,800
3,200,0.073846,0.271746,0.392981,0.776834,0.813199,0.763773,798



=== NMSE TABLE ===


C,20,50,100,200
width,,,,
1,0.883552,0.863957,0.859624,0.888756
2,0.869539,0.836532,0.809430,0.821853
4,0.848645,0.802100,0.778089,0.794538
8,0.857041,0.823303,0.783139,0.784017
16,0.850891,0.822729,0.771138,0.776834



=== EMPIRICAL RAC ===


C,20,50,100,200
rho,,,,
0.20,NaN,NaN,NaN,NaN
0.30,NaN,NaN,NaN,NaN
0.35,NaN,NaN,NaN,NaN
0.40,NaN,NaN,NaN,NaN
0.50,NaN,NaN,NaN,NaN



=== TOP INVERSIONS ===


,width,params,C_easy_abs,C_large_abs,mse_smaller,mse_larger,nmse_harder,nmse_easier,nmse_gap
0,1,78,200,100,0.073846,0.116402,0.888756,0.859624,0.029132
1,1,78,200,50,0.073846,0.161740,0.888756,0.863957,0.024799
2,4,519,200,100,0.073846,0.116402,0.794538,0.778089,0.016449
3,2,189,200,100,0.073846,0.116402,0.821853,0.809430,0.012423
4,16,5523,200,100,0.073846,0.116402,0.776834,0.771138,0.005696
5,1,78,200,20,0.073846,0.206994,0.888756,0.883552,0.005203
6,8,1611,200,100,0.073846,0.116402,0.784017,0.783139,0.000878
